# Suraksha - Model Serving & Real-Time Inference

**Real-time fraud detection inference pipeline for advanced solution**

Person 1's Final Task - Run after 04_multiclass_training

## Capabilities:
- Loads trained model from MLflow Model Registry
- Engineers features for single transaction
- Predicts fraud type with probability
- Returns fraud classification with confidence

**Input:** Single transaction JSON  
**Output:** Fraud prediction with type and probability

## Fraud Types:
0. Legitimate
1. Velocity Fraud
2. Mule Account
3. SIM Swap
4. Device Takeover
5. Beneficiary Manipulation
6. Amount Anomaly
7. Temporal Anomaly
8. Merchant Fraud
9. Failed-Then-Success

In [0]:
import mlflow
import mlflow.pyfunc
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🚀 SURAKSHA - MODEL SERVING & INFERENCE")
print("="*70)

# Fraud type mapping
FRAUD_TYPE_MAP = {
    0: "Legitimate",
    1: "Velocity Fraud",
    2: "Mule Account",
    3: "SIM Swap",
    4: "Device Takeover",
    5: "Beneficiary Manipulation",
    6: "Amount Anomaly",
    7: "Temporal Anomaly",
    8: "Merchant Fraud",
    9: "Failed-Then-Success"
}

In [0]:
%pip install xgboost scikit-learn --quiet
dbutils.library.restartPython()

In [0]:
print("\n📊 Step 1: Loading trained model from pickle file...")

import joblib
import os

# Get current username dynamically
try:
    username = spark.sql("SELECT current_user()").collect()[0][0]
except:
    username = os.environ.get('USER', 'default_user')

print(f"   Current user: {username}")

# Construct model path with dynamic username
model_path = f"/Workspace/Users/{username}/Suraksha/models/suraksha_advanced.pkl"

try:
    # Load model from pickle file
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        print(f"✓ Loaded model from: {model_path}")
        print(f"   Model type: {type(model).__name__}")
    else:
        raise FileNotFoundError(f"Model file not found at: {model_path}")
except Exception as e:
    print(f"❌ Error loading model: {str(e)}")
    print("   Please train the model first using 04_multiclass_training")
    raise

print("✅ Model loaded successfully!")

In [0]:
print("\n📊 Step 2: Defining feature engineering functions...")

import pickle
import os

# FIX 12: Load feature names saved during training
try:
    feature_names_path = f"/Workspace/Users/{username}/Suraksha/models/feature_names.pkl"
    with open(feature_names_path, 'rb') as f:
        MODEL_FEATURES = pickle.load(f)
    print(f"✓ Loaded {len(MODEL_FEATURES)} feature names from training")
except:
    print("⚠️ Using hardcoded feature list as fallback")
    # Fallback: Features the model was trained with (in exact order from training) - 38 features
    MODEL_FEATURES = [
        'sender_bank',
        'sender_state',
        'sender_age_group',
        'device_type',
        'network_type',
        'txn_type',
        'amount_inr',
        'txn_status',
        'mpin_attempts',
        'device_changed_flag',
        'sim_change_recent',
        'sim_age_days',
        'location_changed_flag',
        'receiver_bank',
        'receiver_state',
        'receiver_age_group',
        'merchant_category',
        'high_risk_merchant',
        'hour_of_day',
        'day_of_week',
        'is_weekend',
        'is_odd_hours',
        'amount_is_round',
        'sender_txn_count_1min',
        'sender_txn_count_1hour',
        'sender_txn_count_24h',
        'time_since_last_txn_sec',
        'sender_receiver_history',
        'receiver_inbound_count_1h',
        'receiver_outbound_count_1h',
        'amount_zscore',
        'amount_percentile',
        'amount_zscore_category',
        'amount_percentile_category',
        'weekend_large_txn',
        'merchant_amount_mismatch',
        'failed_count_before',
        'failed_then_success'
    ]

def engineer_features_for_inference(txn_data):
    """
    Engineer features for a single transaction at inference time.
    Matches exactly the features the model was trained with.
    
    Args:
        txn_data: dict with transaction fields
    
    Returns:
        pandas DataFrame with engineered features in correct order
    """
    
    # Convert to DataFrame
    df = pd.DataFrame([txn_data])
    
    # Temporal features (derived from timestamp)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour_of_day'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.day_name()
        df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])
        df['is_odd_hours'] = df['hour_of_day'].isin([23, 0, 1, 2, 3, 4, 5])
    
    # Amount features
    if 'amount_inr' in df.columns:
        df['amount_is_round'] = (df['amount_inr'] % 1000 == 0) | (df['amount_inr'] % 5000 == 0)
        # Z-score and percentile (mock for inference - in production use historical stats)
        df['amount_zscore'] = txn_data.get('amount_zscore', 0.0)
        df['amount_percentile'] = txn_data.get('amount_percentile', 50.0)
        
        # FIX 11: Categorical versions using integers (matching training)
        zscore = df['amount_zscore'].iloc[0]
        if zscore > 3:
            df['amount_zscore_category'] = 3
        elif zscore > 2:
            df['amount_zscore_category'] = 2
        elif zscore > 1:
            df['amount_zscore_category'] = 1
        else:
            df['amount_zscore_category'] = 0
            
        percentile = df['amount_percentile'].iloc[0]
        if percentile > 95:
            df['amount_percentile_category'] = 3
        elif percentile > 90:
            df['amount_percentile_category'] = 2
        elif percentile > 75:
            df['amount_percentile_category'] = 1
        else:
            df['amount_percentile_category'] = 0
    
    # Weekend large transaction
    if 'is_weekend' in df.columns and 'amount_inr' in df.columns:
        df['weekend_large_txn'] = (df['is_weekend'] == True) & (df['amount_inr'] > 15000)
    
    # Merchant amount mismatch (requires category averages - mock for now)
    if 'merchant_category' in df.columns and 'amount_inr' in df.columns:
        df['merchant_amount_mismatch'] = False  # Would check against category averages
    
    # Failed then success pattern
    if 'failed_count_before' in df.columns and 'txn_status' in df.columns:
        df['failed_then_success'] = (df['failed_count_before'] > 0) & (df['txn_status'] == 'SUCCESS')
    
    # Encode categorical variables
    categorical_cols = ['txn_type', 'txn_status', 'sender_bank', 'receiver_bank',
                       'sender_state', 'receiver_state', 'sender_age_group',
                       'receiver_age_group', 'device_type', 'network_type',
                       'day_of_week']
    
    for col in categorical_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    
    # Handle merchant_category (can be None/null)
    if 'merchant_category' in df.columns:
        df['merchant_category'] = df['merchant_category'].fillna('None')
        le = LabelEncoder()
        df['merchant_category'] = le.fit_transform(df['merchant_category'].astype(str))
    
    # Ensure all required features exist and are in correct order
    for feature in MODEL_FEATURES:
        if feature not in df.columns:
            df[feature] = 0  # Default value for missing features
    
    # Return only the features the model expects, in correct order
    return df[MODEL_FEATURES]

print("✓ Feature engineering functions defined")
print(f"   Model expects {len(MODEL_FEATURES)} features")

In [0]:
print("\n📊 Step 3: Defining prediction function...")

def predict_fraud(txn_data):
    """
    Predict fraud type for a single transaction.
    
    Args:
        txn_data: dict with transaction fields
    
    Returns:
        dict with prediction results
    """
    
    # Engineer features (returns DataFrame with correct features in correct order)
    X = engineer_features_for_inference(txn_data)
    
    # Make prediction
    prediction = model.predict(X)[0]
    
    # FIX 13: Get probabilities using predict_proba
    probabilities = model.predict_proba(X)[0]
    
    # Get fraud type
    fraud_type_id = int(prediction)
    fraud_type = FRAUD_TYPE_MAP.get(fraud_type_id, "Unknown")
    
    # Determine if fraudulent
    is_fraud = fraud_type_id > 0
    
    # Calculate confidence score for the predicted class
    confidence_score = float(probabilities[int(prediction)])
    
    result = {
        'transaction_id': txn_data.get('txn_id', 'N/A'),
        'is_fraud': is_fraud,
        'fraud_type': fraud_type,
        'fraud_type_id': fraud_type_id,
        'confidence_score': round(confidence_score * 100, 1),
        'confidence_label': 'High' if confidence_score > 0.80 
                            else 'Medium' if confidence_score > 0.60 
                            else 'Low',
        'all_probabilities': {
            FRAUD_TYPE_MAP[i]: round(float(p) * 100, 1) 
            for i, p in enumerate(probabilities)
        }
    }
    
    return result

print("✓ Prediction function defined")

In [0]:
print("\n📊 Step 4: Testing with example transactions...\n")

# Example 1: Legitimate transaction
legitimate_txn = {
    'txn_id': 'TXN_TEST_001',
    'timestamp': '2024-06-15 14:30:00',
    'amount_inr': 2500,
    'txn_type': 'P2P',
    'txn_status': 'SUCCESS',
    'sender_bank': 'HDFC',
    'receiver_bank': 'ICICI',
    'sender_state': 'Maharashtra',
    'receiver_state': 'Maharashtra',
    'sender_age_group': '26-35',
    'receiver_age_group': '26-35',
    'device_type': 'Android',
    'network_type': '4G',
    'mpin_attempts': 0,
    'device_changed_flag': False,
    'sim_change_recent': False,
    'sim_age_days': 365,
    'location_changed_flag': False,
    'merchant_category': None,
    'high_risk_merchant': False,
    'sender_txn_count_1min': 1,
    'sender_txn_count_1hour': 2,
    'sender_txn_count_24h': 5,
    'time_since_last_txn_sec': 3600,
    'sender_receiver_history': True,
    'receiver_inbound_count_1h': 2,
    'receiver_outbound_count_1h': 1,
    'amount_zscore': 0.5,
    'amount_percentile': 60,
    'failed_count_before': 0
}

result = predict_fraud(legitimate_txn)
print("🟢 Example 1: Legitimate Transaction")
print(f"   Result: {result['fraud_type']}")
print(f"   Is Fraud: {result['is_fraud']}")
print(f"   Confidence: {result['confidence_score']}% ({result['confidence_label']})")
print(f"\n   All Probabilities:")
for fraud_type, prob in result['all_probabilities'].items():
    if prob > 1.0:  # Only show types with >1% probability
        print(f"      {fraud_type}: {prob}%")

In [0]:
# Example 2: Amount Anomaly Fraud (Very High Amount)
amount_anomaly_txn = {
    'txn_id': 'TXN_TEST_002',
    'timestamp': '2024-06-15 14:30:00',
    'amount_inr': 95000,  # Very high amount!
    'txn_type': 'P2P',
    'txn_status': 'SUCCESS',
    'sender_bank': 'HDFC',
    'receiver_bank': 'Paytm',
    'sender_state': 'Delhi',
    'receiver_state': 'Karnataka',
    'sender_age_group': '18-25',
    'receiver_age_group': '26-35',
    'device_type': 'Android',
    'network_type': '4G',
    'mpin_attempts': 0,
    'device_changed_flag': False,
    'sim_change_recent': False,
    'sim_age_days': 180,
    'location_changed_flag': False,
    'merchant_category': None,
    'high_risk_merchant': False,
    'sender_txn_count_1min': 1,
    'sender_txn_count_1hour': 2,
    'sender_txn_count_24h': 3,
    'time_since_last_txn_sec': 300,
    'sender_receiver_history': False,
    'receiver_inbound_count_1h': 1,
    'receiver_outbound_count_1h': 0,
    'amount_zscore': 8.5,  # Extremely high Z-score!
    'amount_percentile': 99.5,  # Top 0.5%
    'failed_count_before': 0
}

result = predict_fraud(amount_anomaly_txn)
print("\n🔴 Example 2: Amount Anomaly Fraud")
print(f"   Result: {result['fraud_type']}")
print(f"   Is Fraud: {result['is_fraud']}")
print(f"   Confidence: {result['confidence_score']}% ({result['confidence_label']})")
print(f"\n   All Probabilities:")
for fraud_type, prob in result['all_probabilities'].items():
    if prob > 1.0:  # Only show types with >1% probability
        print(f"      {fraud_type}: {prob}%")

In [0]:
# Example 3: Mule Account Fraud (High Inbound Activity)
mule_account_txn = {
    'txn_id': 'TXN_TEST_003',
    'timestamp': '2024-06-15 16:45:00',
    'amount_inr': 15000,
    'txn_type': 'P2P',
    'txn_status': 'SUCCESS',
    'sender_bank': 'SBI',
    'receiver_bank': 'Paytm',
    'sender_state': 'Maharashtra',
    'receiver_state': 'Uttar Pradesh',
    'sender_age_group': '18-25',  # Young sender
    'receiver_age_group': '18-25',  # Young receiver
    'device_type': 'Android',
    'network_type': '4G',
    'mpin_attempts': 0,
    'device_changed_flag': False,
    'sim_change_recent': False,
    'sim_age_days': 90,
    'location_changed_flag': False,
    'merchant_category': None,
    'high_risk_merchant': False,
    'sender_txn_count_1min': 1,
    'sender_txn_count_1hour': 2,
    'sender_txn_count_24h': 5,
    'time_since_last_txn_sec': 200,
    'sender_receiver_history': False,  # First time transaction
    'receiver_inbound_count_1h': 15,  # Very high inbound!
    'receiver_outbound_count_1h': 12,  # High outbound too!
    'amount_zscore': 2.8,
    'amount_percentile': 85,
    'failed_count_before': 0
}

result = predict_fraud(mule_account_txn)
print("\n🔴 Example 3: Mule Account Fraud")
print(f"   Result: {result['fraud_type']}")
print(f"   Is Fraud: {result['is_fraud']}")
print(f"   Confidence: {result['confidence_score']}% ({result['confidence_label']})")
print(f"\n   All Probabilities:")
for fraud_type, prob in result['all_probabilities'].items():
    if prob > 1.0:  # Only show types with >1% probability
        print(f"      {fraud_type}: {prob}%")

print("\n" + "="*70)
print("✅ MODEL SERVING PIPELINE READY!")
print("="*70)
print("\n📊 Next Steps:")
print("   1. Deploy this as an API endpoint")
print("   2. Integrate with RAG for fraud explanations (shared/rag_pipeline)")
print("   3. Build Streamlit/Gradio app for demo")
print("   4. Connect to real-time transaction stream")
print("\n" + "="*70)